# 01 — Copy（数据拷贝）

## 背景

GPU 的内存带宽是深度学习训练与推理的核心瓶颈之一。现代 GPU（如 H100）拥有高达 3.35 TB/s 的 HBM 带宽，但只有充分利用并行度，才能榨干这一带宽。

数据拷贝是最基础的内存操作，也是理解 GPU 编程模型的起点：
- **Block（块）**：GPU 上并行执行的独立工作单元
- **Thread（线程）**：Block 内的最细粒度执行单元，每个线程处理一部分数据

## 问题定义

将张量 `A` 的全部元素复制到张量 `B`：

$$B[i] = A[i], \quad i = 0, 1, \ldots, N-1$$

TileLang 展示三种实现，性能逐步提升：

| 实现 | Block 数 | 线程数/Block | 说明 |
|------|----------|-------------|------|
| 串行 | 1 | 1 | 单线程顺序执行 |
| 多线程 | 1 | 256 | 块内 256 线程并行 |
| 多块并行 | N/BLOCK_N | 256 | 多块 + 多线程全并行 |

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
N = 1024 * 256
A = torch.randn(N, dtype=torch.float16, device="cuda")

def ref_copy(A: torch.Tensor) -> torch.Tensor:
    return A.clone()

B = ref_copy(A)
assert torch.allclose(A, B)
print(f"输入形状: {A.shape}, dtype: {A.dtype}")
print("PyTorch 参考实现正确 ✅")

## Triton 实现

Triton 使用 `@triton.jit` 装饰器，通过 `tl.program_id` 获取当前 Block 编号，`tl.arange` 生成连续下标，`tl.load` / `tl.store` 完成内存访问，`mask` 参数处理边界。

In [ ]:
@triton.jit
def _copy_kernel(A_ptr, B_ptr, N, BLOCK: tl.constexpr):
    pid  = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < N
    tl.store(B_ptr + offs, tl.load(A_ptr + offs, mask=mask), mask=mask)

def triton_copy(A: torch.Tensor) -> torch.Tensor:
    B = torch.empty_like(A)
    N = A.numel()
    BLOCK = 1024
    grid = (triton.cdiv(N, BLOCK),)
    _copy_kernel[grid](A, B, N, BLOCK=BLOCK)
    return B

# 验证
B_tri = triton_copy(A)
print("Triton 正确性验证:", "✅ PASSED" if torch.allclose(A, B_tri) else "❌ FAILED")

## TileLang 实现

TileLang 使用 `@tilelang.jit` 装饰器，`T.Kernel` 声明网格维度，`T.copy` 自动并行并向量化内存拷贝。

In [ ]:
# 串行（1 块 × 1 线程）
@tilelang.jit
def tl_copy_serial(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=1):
        T.copy(A, B)
    return B

# 多线程（1 块 × 256 线程）
@tilelang.jit
def tl_copy_multithread(A):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(1, threads=256):
        T.copy(A, B)
    return B

# 多块并行（N//BLOCK_N 块 × 256 线程）
@tilelang.jit
def tl_copy_parallel(A, BLOCK_N: int):
    N = T.const("N")
    A: T.Tensor((N,), T.float16)
    B = T.empty((N,), T.float16)
    with T.Kernel(N // BLOCK_N, threads=256) as pid:
        T.copy(A[pid * BLOCK_N:(pid + 1) * BLOCK_N],
               B[pid * BLOCK_N:(pid + 1) * BLOCK_N])
    return B

# 正确性验证
def check(name, tl_fn, hyper):
    k = tl_fn.compile(**hyper)
    ok = torch.allclose(A, k(A.clone()))
    print(f"  {name}: {'✅ PASSED' if ok else '❌ FAILED'}")

check("串行   ", tl_copy_serial,     {"N": N})
check("多线程 ", tl_copy_multithread, {"N": N})
check("多块并行", tl_copy_parallel,   {"N": N, "BLOCK_N": 1024})

## 性能对比

对比 PyTorch、Triton 与三种 TileLang 实现的延迟，直观展示并行度对性能的影响。

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

k_serial = tl_copy_serial.compile(N=N)
k_mt     = tl_copy_multithread.compile(N=N)
k_par    = tl_copy_parallel.compile(N=N, BLOCK_N=1024)
inp = A.clone()

results = {
    "PyTorch": bench(ref_copy, [inp]),
    "Triton":  bench(triton_copy, [inp]),
    "TileLang\n串行":    bench(k_serial, [inp]),
    "TileLang\n多线程":  bench(k_mt,    [inp]),
    "TileLang\n多块并行": bench(k_par,  [inp]),
}

for name, ms in results.items():
    bw = N * 2 * 2 / ms / 1e9   # read + write, float16
    print(f"  {name.replace(chr(10),' '):22s}: {ms:.4f} ms  ({bw:.2f} TB/s)")

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52", "#8172B2"]
labels = list(results.keys())
values = list(results.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(labels, values, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("延迟 (ms)"); ax.set_title(f"Copy 延迟对比  (N={N:,}, fp16)")
ax.set_ylim(0, max(values)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bws = [N*2*2/ms/1e9 for ms in values]
bars = ax.bar(labels, bws, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, bws):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(bws)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("有效带宽 (TB/s)"); ax.set_title(f"Copy 带宽利用对比  (N={N:,}, fp16)")
ax.set_ylim(0, max(bws)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()